In [ ]:
import openai, json

client = openai.OpenAI()
messages = []

In [ ]:
# import urllib.request
import requests, json

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


def get_popular_movies():
    responses = requests.get(f"{BASE_URL}/movies")
    return json.dumps(responses.json())

def get_movie_details(id):
    response = requests.get(f"{BASE_URL}/movies/{id}")
    return json.dumps(response.json())

def get_movie_credits(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/credits")
    return json.dumps(response.json())

def get_movie_similar(id):
    response = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return json.dumps(response.json())

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_movie_similar": get_movie_similar,
}

# print(get_popular_movies())

In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기 있는 영화 목록",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "ID로 영화 상세 정보 조회",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "영화 ID"},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "영화의 출연진 및 제작진 조회",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "영화 ID"},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "영화의 유사 영화 조회",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "string", "description": "영화 ID"},
                },
                "required": ["id"],
            },
        },
    },
]

In [ ]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화가 무엇인지 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {} for a result of [{"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/5lTZyuBTNOfawsfPT8Q0cIg6qAF.jpg", "genre_ids": [27, 53], "id": 1339713, "title": "Obsession", "original_language": "en", "original_title": "Obsession", "overview": "After breaking the mysterious \"One Wish Willow\" to win his crush's heart, a hopeless romantic finds himself getting exactly what he asked for but soon discovers that some desires come at a dark, sinister price.", "popularity": 917.7069, "poster_path": "https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg", "release_date": "2026-05-13", "softcore": false, "video": false, "vote_average": 7.926, "vote_count": 617}, {"adult": false, "backdrop_path": "https://image.tmdb.org/t/p/w1280/oPsRr7AfNLw6XaPuMpvkWK0bIUA.jpg", "genre_ids": [28, 18], "id": 1057265, "title": "Peddi", "original_language": "te", "original_title": "\u0c2a\u